## Gera tabela para identificar ações  não enviadas

In [24]:
# Célula 1 — imports e config
from pyspark.sql import functions as F, Window

SQL_ENDPOINT = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

jdbc_url = (
    'jdbc:sqlserver://' + SQL_ENDPOINT
    + ';database=wh_siplan_rps'
    + ';encrypt=true;trustServerCertificate=false'
)

token = mssparkutils.credentials.getToken('https://database.windows.net/.default')

# Célula 2 — lê as tabelas
df_lake = spark.sql("SELECT atividade_id, autonomia, PrimeiraData FROM lake_gold_fatos.dbo.base")

# CORRIGIDO: lê do warehouse via JDBC
df_ware = (
    spark.read
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', 'dbo.analise_siplan_rps')
    .option('accessToken', token)
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .load()
    .select("atividade_id", "autonomia", "Status")
)

StatementMeta(, 2cbadad5-b8b5-4e73-b501-5ee457065e6f, 29, Finished, Available, Finished, False)

In [25]:
# Célula 3 — filtra o lakehouse (atividades do ano corrente em diante)
df_filtrado = df_lake.filter(
    F.year(F.col("PrimeiraData")) >= F.year(F.current_date())
).select(
    "atividade_id",
    "autonomia",
    F.col("PrimeiraData").cast("date").alias("PrimeiraData")  # força DATE
)

StatementMeta(, 2cbadad5-b8b5-4e73-b501-5ee457065e6f, 30, Finished, Available, Finished, False)

In [26]:
# Célula 4 — join + status + campos de prazo
dias = F.datediff(F.col("PrimeiraData"), F.current_date())  # positivo = futuro

df_status = (
    df_filtrado.join(df_ware, on="atividade_id", how="left")
    .select(
        df_filtrado["atividade_id"],
        F.when(df_ware["atividade_id"].isNotNull(), df_ware["autonomia"])
         .when(df_filtrado["autonomia"].isNotNull(), df_filtrado["autonomia"])
         .otherwise(F.lit("UO")).alias("autonomia"),
        F.when(df_ware["atividade_id"].isNotNull(), df_ware["Status"])
         .otherwise(F.lit("Não enviada")).alias("StatusGeral"),
        df_filtrado["PrimeiraData"],
        dias.alias("dias_para_inicio"),
        F.when(dias < 0,   F.lit("Já foi"))
         .when(dias <= 10, F.lit("≤10d"))
         .when(dias <= 20, F.lit("≤20d"))
         .when(dias <= 30, F.lit("≤30d"))
         .when(dias <= 45, F.lit("≤45d"))
         .when(dias <= 60, F.lit("≤60d"))
         .otherwise(       F.lit("60d+"))
         .alias("faixa_prazo"),
        F.when(dias < 0,   F.lit(10))
         .when(dias <= 10, F.lit(1))
         .when(dias <= 20, F.lit(2))
         .when(dias <= 30, F.lit(3))
         .when(dias <= 45, F.lit(4))
         .when(dias <= 60, F.lit(5))
         .otherwise(       F.lit(6))
         .alias("ordem_faixa_prazo"),
    )
    # Regra de negócio: autonomia UO sempre mantém StatusGeral = AutonomiaUO
    .withColumn("StatusGeral",
        F.when(F.col("autonomia") == "UO", F.lit("AutonomiaUO"))
         .otherwise(F.col("StatusGeral"))
    )
    .withColumn("ordem_status",
        F.when(F.col("StatusGeral") == "Não enviada",       F.lit(1))
         .when(F.col("StatusGeral") == "Em revisão",        F.lit(2))
         .when(F.col("StatusGeral") == "Doc Gerado",        F.lit(3))
         .when(F.col("StatusGeral") == "Autorizada",        F.lit(4))
         .when(F.col("StatusGeral") == "Enviada",           F.lit(5))
         .when(F.col("StatusGeral") == "Reenviada",         F.lit(6))
         .when(F.col("StatusGeral") == "Em análise",        F.lit(7))
         .when(F.col("StatusGeral") == "Análise gerencial", F.lit(8))
         .when(F.col("StatusGeral") == "Validada",          F.lit(9))
         .when(F.col("StatusGeral") == "Impressa",          F.lit(10))
         .when(F.col("StatusGeral") == "AguardaSTS",        F.lit(11))
         .when(F.col("StatusGeral") == "AguardaDIREG",      F.lit(12))
         .when(F.col("StatusGeral") == "Cancelada",         F.lit(13))
         .when(F.col("StatusGeral") == "Indeferida",        F.lit(14))
         .when(F.col("StatusGeral") == "AutonomiaUO",       F.lit(15))
         .otherwise(F.lit(None))
    )
)

df_status.printSchema()


StatementMeta(, 2cbadad5-b8b5-4e73-b501-5ee457065e6f, 31, Finished, Available, Finished, False)

root
 |-- atividade_id: double (nullable = true)
 |-- autonomia: string (nullable = true)
 |-- StatusGeral: string (nullable = true)
 |-- PrimeiraData: date (nullable = true)
 |-- dias_para_inicio: integer (nullable = true)
 |-- faixa_prazo: string (nullable = false)
 |-- ordem_faixa_prazo: integer (nullable = false)
 |-- ordem_status: integer (nullable = true)



In [31]:
# Célula 4.5 — materializa e verifica ANTES de gravar
df_status.cache()
n = df_status.count()
print(f"Linhas no df_status: {n}")
# df_status.show(5)

if n == 0:
    raise ValueError("df_status está vazio — verifique as células anteriores")

StatementMeta(, 2cbadad5-b8b5-4e73-b501-5ee457065e6f, 38, Finished, Available, Finished, False)

Linhas no df_status: 38477


In [ ]:
# Célula 5 — grava E registra no catálogo (aparece dentro do dbo)
df_status.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lake_gold_fatos.dbo.status_do_fluxo")

print("OK — status_do_fluxo registrada em lake_gold_fatos.dbo")
